# 🧬 CryoAtom2: From cryo-EM density map to atomic structure

CryoAtom2 is a software tool that automatically constructs all-atom models of proteins, nucleic acids, or their complexes from cryo-EM density maps and sequence information.

<div align="center">
  <img src="https://raw.githubusercontent.com/YangLab-SDU/CryoAtom/master/imgs/framework.png" width="600">
</div>

Paper available in [**Nature Structural & Molecular Biology**](https://www.nature.com/articles/s41594-025-01713-3).If you encounter any bugs, please report the issue to [**GitHub Issues**](https://github.com/YangLab-SDU/CryoAtom/issues).

[**YangLab-SDU**](https://yanglab.qd.sdu.edu.cn) focuses on computational biology and structural bioinformatics. Follow us on [**GitHub**](https://github.com/YangLab-SDU).

---

### 🚀 Quick Start (Execute steps sequentially: 1 → 2 → 3)

1. **Configure (Step 1):** Enter **Job Name** and upload/path files. Click **`CONFIRM JOB`**.
   *(Note: A dedicated folder `/content/[Job Name]` will be created for your results.)*
2. **Setup (Step 2):** Run once per session. Choose whether to use **Google Drive** for faster weight loading or download manually.
3. **Run (Step 3):** Start the modeling process.
4. **Visualize (Step 4):** **Preview the predicted structure.** After Step 3 completes, run the fourth code block to interactively visualize the 3D model directly at the bottom of the page.

**💡 Tips:**
- **Use GPU:** Before running the pipeline, please check that the runtime type is set to GPU at **"Runtime"** -> **"Change runtime type"**.
- **Output Files:** The results will include **[Job Name].cif** and **[Job Name]_raw.cif**, representing the predicted results and the results without confidence filtering, respectively.
- **New Tasks:** Simply update Step 1 and run Step 3 (skip Step 2). Once finished, run Step 4 to refresh and view the new predicted structure.

In [1]:
#@title # Step 1:  Configure Inference & Data Upload { display-mode: "form" }
import os
import base64
import subprocess
import requests
import urllib3
from google.colab import output
from IPython.display import HTML, display

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

if 'final_cmd' not in globals():
    final_cmd = ""
command_ready = False
EXAMPLE_DIR = "/content/Example_7xht"
ZIP_URL = "https://yanglab.qd.sdu.edu.cn/CryoAtom/download/7xht.zip"

def download_example_data():
    if not os.path.exists(EXAMPLE_DIR):
        os.makedirs(EXAMPLE_DIR, exist_ok=True)

    target_map = os.path.join(EXAMPLE_DIR, "emd_33198.map")
    if os.path.exists(target_map):
        return

    output.eval_js("setUIProgress(0, 'Connecting to server...', true)")
    zip_path = os.path.join(EXAMPLE_DIR, "7xht.zip")

    try:
        response = requests.get(ZIP_URL, stream=True, verify=False, timeout=60)
        response.raise_for_status()
        total_size = int(response.headers.get('content-length', 0))

        downloaded = 0
        with open(zip_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)
                    if total_size > 0:
                        p = int((downloaded / total_size) * 100)
                        output.eval_js(f"setUIProgress({p}, 'Downloading 7xht.zip: {p}%', true)")

        output.eval_js("setUIProgress(100, 'Extracting files...', true)")
        result = subprocess.run(f"unzip -o {zip_path} -d {EXAMPLE_DIR}", shell=True, capture_output=True, text=True)
        if result.returncode == 0:
            os.remove(zip_path)
            output.eval_js("setUIProgress(100, 'Example data ready!', true)")
        else:
            output.eval_js("setUIProgress(0, 'Extraction failed!', true)")

    except Exception as e:
        output.eval_js(f"setUIProgress(0, 'Error: {str(e)[:40]}', true)")

    output.eval_js("setTimeout(() => { document.getElementById('progress_container').style.display = 'none'; }, 2000)")

html_code = f"""
<div id="cryo-portal" style="padding: 25px; background: #ffffff; border: 1px solid #e0e0e0; border-radius: 12px; font-family: 'Segoe UI', Arial, sans-serif; box-shadow: 0 4px 15px rgba(0,0,0,0.05); max-width: 1000px;">
    <h2 style="color: #1a73e8; margin-top: 0; border-bottom: 2px solid #e8f0fe; padding-bottom: 10px;">CryoAtom Configuration Portal</h2>

<div style="margin-bottom: 20px;">
    <label style="font-weight: bold; color: #3c4043;">Job Name (Task Folder):</label><br>
    <input type="text" id="job_name" value="My_CryoAtom_Task" style="width: 100%; padding: 10px; margin-top: 8px; border: 1px solid #dadce0; border-radius: 6px; box-sizing: border-box;">
</div>

<div style="margin-bottom: 25px; padding: 10px; background: #f1f8e9; border-radius: 6px;">
    <label style="cursor: pointer; color: #2e7d32; font-weight: bold;">
        <input type="checkbox" id="is_test" onchange="toggleTest(this)"> Use Example Test Data (7xht)
    </label>
</div>

<div id="progress_container" style="display: none; margin-bottom: 20px; padding: 15px; background: #f8f9fa; border-radius: 8px; border: 1px solid #1a73e8;">
    <div style="display: flex; justify-content: space-between; margin-bottom: 5px;">
        <span id="status_msg" style="font-size: 13px; color: #1a73e8; font-weight: bold;">Ready</span>
        <span id="progress_percent" style="font-size: 13px; color: #666;">0%</span>
    </div>
    <div style="width: 100%; background-color: #e0e0e0; border-radius: 10px; height: 10px; overflow: hidden;">
        <div id="progress_bar" style="width: 0%; height: 100%; background-color: #2ecc71; transition: width 0.1s;"></div>
    </div>
</div>

<h4 style="color: #5f6368; margin-bottom: 10px; border-left: 4px solid #1a73e8; padding-left: 10px;">1. Map & Sequences Input</h4>
<div style="display: grid; grid-template-columns: 1fr 1fr; gap: 15px; margin-bottom: 20px;">
    <div>
        <label style="font-size: 13px; font-weight: 600;">Density Map (.map/.mrc) <span style="color:red">*</span></label>
        <div style="display: flex; margin-top: 5px;">
            <input type="text" id="map_path" placeholder="input map file" style="flex: 1; padding: 8px; border: 1px solid #dadce0; border-radius: 4px 0 0 4px;">
            <button onclick="clickHidden('file_map')" id="btn_map" style="background: #1a73e8; color: white; border: none; padding: 0 12px; border-radius: 0 4px 4px 0; cursor: pointer;">Upload</button>
        </div>
        <input type="file" id="file_map" style="display:none" onchange="startChunkedUpload(this, 'map_path')">
    </div>
    <div>
        <label style="font-size: 13px; font-weight: 600;">Protein Sequence (.fasta)</label>
        <div style="display: flex; margin-top: 5px;">
            <input type="text" id="prot_path" placeholder="input protein fasta" style="flex: 1; padding: 8px; border: 1px solid #dadce0; border-radius: 4px 0 0 4px;">
            <button onclick="clickHidden('file_prot')" id="btn_prot" style="background: #1a73e8; color: white; border: none; padding: 0 12px; border-radius: 0 4px 4px 0; cursor: pointer;">Upload</button>
        </div>
        <input type="file" id="file_prot" style="display:none" onchange="startChunkedUpload(this, 'prot_path')">
    </div>
    <div>
        <label style="font-size: 13px; font-weight: 600;">RNA Sequence (.fasta)</label>
        <div style="display: flex; margin-top: 5px;">
            <input type="text" id="rna_path" placeholder="input RNA fasta" style="flex: 1; padding: 8px; border: 1px solid #dadce0; border-radius: 4px 0 0 4px;">
            <button onclick="clickHidden('file_rna')" id="btn_rna" style="background: #1a73e8; color: white; border: none; padding: 0 12px; border-radius: 0 4px 4px 0; cursor: pointer;">Upload</button>
        </div>
        <input type="file" id="file_rna" style="display:none" onchange="startChunkedUpload(this, 'rna_path')">
    </div>
    <div>
        <label style="font-size: 13px; font-weight: 600;">DNA Sequence (.fasta)</label>
        <div style="display: flex; margin-top: 5px;">
            <input type="text" id="dna_path" placeholder="input DNA fasta" style="flex: 1; padding: 8px; border: 1px solid #dadce0; border-radius: 4px 0 0 4px;">
            <button onclick="clickHidden('file_dna')" id="btn_dna" style="background: #1a73e8; color: white; border: none; padding: 0 12px; border-radius: 0 4px 4px 0; cursor: pointer;">Upload</button>
        </div>
        <input type="file" id="file_dna" style="display:none" onchange="startChunkedUpload(this, 'dna_path')">
    </div>
</div>

<h4 style="color: #5f6368; margin-bottom: 10px; border-left: 4px solid #34a853; padding-left: 10px;">2. Sequence Databases</h4>
<div style="display: grid; grid-template-columns: 1fr 1fr; gap: 15px; margin-bottom: 20px;">
    <div>
        <label style="font-size: 13px; font-weight: 600;">Protein Database (.fasta)</label>
        <div style="display: flex; margin-top: 5px;">
            <input type="text" id="pf_path" placeholder="Protein DB" style="flex: 1; padding: 8px; border: 1px solid #dadce0; border-radius: 4px 0 0 4px;">
            <button onclick="clickHidden('file_pf')" style="background: #34a853; color: white; border: none; padding: 0 12px; border-radius: 0 4px 4px 0; cursor: pointer;">Upload</button>
        </div>
        <input type="file" id="file_pf" style="display:none" onchange="startChunkedUpload(this, 'pf_path')">
    </div>
    <div>
        <label style="font-size: 13px; font-weight: 600;">Nucleic Acid Database (.fasta)</label>
        <div style="display: flex; margin-top: 5px;">
            <input type="text" id="nf_path" placeholder="Nucleic Acid DB" style="flex: 1; padding: 8px; border: 1px solid #dadce0; border-radius: 4px 0 0 4px;">
            <button onclick="clickHidden('file_nf')" style="background: #34a853; color: white; border: none; padding: 0 12px; border-radius: 0 4px 4px 0; cursor: pointer;">Upload</button>
        </div>
        <input type="file" id="file_nf" style="display:none" onchange="startChunkedUpload(this, 'nf_path')">
    </div>
</div>

<h4 style="color: #5f6368; margin-bottom: 10px; border-left: 4px solid #fbbc05; padding-left: 10px;">3. Local Modeling & Identification</h4>
<div style="display: grid; grid-template-columns: 1fr 1fr; gap: 15px; margin-bottom: 25px;">
    <div>
        <label style="font-size: 13px; font-weight: 600;">Mask Map (.map/.mrc)</label>
        <div style="display: flex; margin-top: 5px;">
            <input type="text" id="m_path" placeholder="Optional mask" style="flex: 1; padding: 8px; border: 1px solid #dadce0; border-radius: 4px 0 0 4px;">
            <button onclick="clickHidden('file_m')" style="background: #fbbc05; color: black; border: none; padding: 0 12px; border-radius: 0 4px 4px 0; cursor: pointer;">Upload</button>
        </div>
        <input type="file" id="file_m" style="display:none" onchange="startChunkedUpload(this, 'm_path')">
    </div>
    <div>
        <label style="font-size: 13px; font-weight: 600;">Backbone Atoms (backbone.cif)</label>
        <div style="display: flex; margin-top: 5px;">
            <input type="text" id="r_path" placeholder="Protein-NA backbone" style="flex: 1; padding: 8px; border: 1px solid #dadce0; border-radius: 4px 0 0 4px;">
            <button onclick="clickHidden('file_r')" style="background: #fbbc05; color: black; border: none; padding: 0 12px; border-radius: 0 4px 4px 0; cursor: pointer;">Upload</button>
        </div>
        <input type="file" id="file_r" style="display:none" onchange="startChunkedUpload(this, 'r_path')">
    </div>
</div>

<button onclick="saveParams()" id="confirm_btn" style="width: 100%; padding: 16px; background: #1a73e8; color: #fff; border: none; border-radius: 8px; font-size: 18px; font-weight: bold; cursor: pointer; box-shadow: 0 4px 6px rgba(26,115,232,0.3); transition: background 0.3s;">CONFIRM JOB</button>
</div>

<script>
    window.setUIProgress = function(percent, message, show) {{
        const container = document.getElementById('progress_container');
        const bar = document.getElementById('progress_bar');
        const msg = document.getElementById('status_msg');
        const pct = document.getElementById('progress_percent');

        if (show) container.style.display = 'block';
        bar.style.width = percent + '%';
        msg.innerText = message;
        pct.innerText = percent + '%';
    }}

    function toggleTest(cb) {{
        const jobInput = document.getElementById('job_name');
        const paths = {{
            'map_path': '{EXAMPLE_DIR}/emd_33198.map',
            'prot_path': '{EXAMPLE_DIR}/protein.fasta',
            'rna_path': '{EXAMPLE_DIR}/rna.fasta',
            'dna_path': '{EXAMPLE_DIR}/dna.fasta'
        }};

        if(cb.checked) {{
            jobInput.value = "Example_7xht";
            jobInput.disabled = true;
            jobInput.style.backgroundColor = "#f0f0f0";
            for (let id in paths) document.getElementById(id).value = paths[id];
        }} else {{
            jobInput.disabled = false;
            jobInput.style.backgroundColor = "#fff";
            jobInput.value = "My_CryoAtom_Task";
            for (let id in paths) document.getElementById(id).value = "";
        }}
    }}

    function clickHidden(id) {{ document.getElementById(id).click(); }}

    async function startChunkedUpload(input, targetId) {{
        const file = input.files[0];
        if (!file) return;

        const jobName = document.getElementById('job_name').value;
        setUIProgress(0, "Streaming: " + file.name, true);

        const chunkSize = 1024 * 1024;
        const totalChunks = Math.ceil(file.size / chunkSize);

        await google.colab.kernel.invokeFunction('notebook.InitUpload', [file.name, jobName], {{}});

        for (let i = 0; i < totalChunks; i++) {{
            const start = i * chunkSize;
            const end = Math.min(file.size, start + chunkSize);
            const chunk = file.slice(start, end);

            const base64Chunk = await new Promise((resolve) => {{
                const reader = new FileReader();
                reader.onload = () => resolve(reader.result.split(',')[1]);
                reader.readAsDataURL(chunk);
            }});

            await google.colab.kernel.invokeFunction('notebook.WriteChunk', [base64Chunk], {{}});
            const p = Math.round(((i + 1) / totalChunks) * 100);
            setUIProgress(p, "Streaming: " + file.name, true);
        }}

        await google.colab.kernel.invokeFunction('notebook.FinalizeUpload', [targetId], {{}});
        setUIProgress(100, "Upload Complete!", true);
        setTimeout(() => {{ document.getElementById('progress_container').style.display = 'none'; }}, 2000);
        input.value = "";
    }}

    window.setPathValue = function(inputId, path) {{
        document.getElementById(inputId).value = path;
    }}

    function saveParams() {{
        const params = {{
            o:   document.getElementById('job_name').value,
            v:   document.getElementById('map_path').value,
            ps:  document.getElementById('prot_path').value,
            rs:  document.getElementById('rna_path').value,
            ds:  document.getElementById('dna_path').value,
            pf:  document.getElementById('pf_path').value,
            nf:  document.getElementById('nf_path').value,
            m:   document.getElementById('m_path').value,
            r:   document.getElementById('r_path').value,
            is_test: document.getElementById('is_test').checked
        }};
        google.colab.kernel.invokeFunction('notebook.BuildCommand', [params], {{}});
    }}
</script>
"""

def init_upload(filename, job_name):
    global current_file_path
    job_dir = f"/content/{job_name}"
    os.makedirs(job_dir, exist_ok=True)
    current_file_path = os.path.join(job_dir, filename)
    with open(current_file_path, "wb") as f:
        pass

def write_chunk(base64_chunk):
    global current_file_path
    with open(current_file_path, "ab") as f:
        f.write(base64.b64decode(base64_chunk))

def finalize_upload(target_id):
    global current_file_path
    output.eval_js(f"window.setPathValue('{target_id}', '{current_file_path}')")
    return current_file_path

def build_command(p):
    global final_cmd, command_ready

    if p.get('is_test'):
        download_example_data()

    if not p['v']:
        output.eval_js("alert('Density Map is required!')")
        return

    job_dir = f"/content/{p['o']}"
    os.makedirs(job_dir, exist_ok=True)
    script = "/content/CryoAtom/CryoAtom2/build.py"
    cmd = f"python {script} -v {p['v']} -o {job_dir} -d 0"
    mapping = {'ps':'-ps', 'rs':'-rs', 'ds':'-ds', 'pf':'-pf', 'nf':'-nf', 'm':'-m', 'r':'-r'}
    for key, flag in mapping.items():
        if p.get(key):
            cmd += f" {flag} {p[key]}"

    final_cmd = cmd
    command_ready = True
    output.eval_js("document.getElementById('confirm_btn').style.background = '#f39c12'; document.getElementById('confirm_btn').innerText = 'JOB CONFIGURED';")

output.register_callback('notebook.InitUpload', init_upload)
output.register_callback('notebook.WriteChunk', write_chunk)
output.register_callback('notebook.FinalizeUpload', finalize_upload)
output.register_callback('notebook.BuildCommand', build_command)

display(HTML(html_code))

In [3]:
#@title # Step 2: Install dependencies (~6mins) { display-mode: "form" }
#@markdown ---
#@markdown ### Model Weight Configuration:
Use_Google_Drive = False #@param {type:"boolean"}
#@markdown * **If checked:** Please add the folder at this [link](https://drive.google.com/drive/folders/1BUWOsEXYyX85ClZvNjzeTCaTKQd9UVF9?usp=drive_link) as a **"Shortcut"** to your Google Drive first. This will reduce setup time.
#@markdown * **If unchecked:** The weights will be automatically downloaded to the required location during execution.
#@markdown ---
import os
import sys
import base64
import subprocess
import importlib.util
import importlib.metadata
from google.colab import drive, output
from IPython.display import HTML, display

# --- UI Component ---
progress_html = """
<div id="setup-portal" style="padding: 20px; background: #ffffff; border: 1px solid #e0e0e0; border-radius: 12px; font-family: 'Segoe UI', Arial, sans-serif; box-shadow: 0 4px 15px rgba(0,0,0,0.05); max-width: 1000px;">
    <h3 style="color: #1a73e8; margin-top: 0; border-bottom: 2px solid #e8f0fe; padding-bottom: 10px;">CryoAtom Environment Setup</h3>
    <div id="setup_progress_container" style="margin-top: 15px; padding: 15px; background: #f8f9fa; border-radius: 8px; border: 1px solid #1a73e8;">
        <div style="display: flex; justify-content: space-between; margin-bottom: 5px;">
            <span id="setup_status_msg" style="font-size: 13px; color: #1a73e8; font-weight: bold;">Initializing...</span>
            <span id="setup_progress_percent" style="font-size: 13px; color: #666;">0%</span>
        </div>
        <div style="width: 100%; background-color: #e0e0e0; border-radius: 10px; height: 10px; overflow: hidden;">
            <div id="setup_progress_bar" style="width: 0%; height: 100%; background-color: #2ecc71; transition: width 0.3s;"></div>
        </div>
    </div>
</div>
<script>
    window.updateSetupProgress = function(percent, message, color='#2ecc71') {
        const bar = document.getElementById('setup_progress_bar');
        const msg = document.getElementById('setup_status_msg');
        const pct = document.getElementById('setup_progress_percent');
        bar.style.width = percent + '%';
        bar.style.backgroundColor = color;
        msg.innerText = message;
        pct.innerText = percent + '%';
    }
</script>
"""
display(HTML(progress_html))

def set_progress(p, m, c='#2ecc71'):
    output.eval_js(f"updateSetupProgress({p}, '{m}', '{c}')")

# --- Helper Functions ---
def get_installed_version(package_name):
    try: return importlib.metadata.version(package_name)
    except importlib.metadata.PackageNotFoundError: return None

def check_package_status(import_name, pkg_with_version):
    pkg_name = pkg_with_version.split('==')[0]
    required_version = pkg_with_version.split('==')[1] if '==' in pkg_with_version else None
    spec = importlib.util.find_spec(import_name)
    if spec is None: return 1
    if required_version:
        installed_version = get_installed_version(pkg_name)
        if installed_version != required_version: return 2
    return 0

def smart_install(import_name, pkg_with_version):
    status = check_package_status(import_name, pkg_with_version)
    pkg_name = pkg_with_version.split('==')[0]
    required_version = pkg_with_version.split('==')[1] if '==' in pkg_with_version else None
    if status == 0: return
    cmd = f"pip install {pkg_with_version} -q"
    process = subprocess.run(cmd, shell=True, capture_output=True)
    if process.returncode != 0 and required_version:
        cmd_smooth = f"pip install '{pkg_name}>={required_version}' -q"
        subprocess.run(cmd_smooth, shell=True, capture_output=True)

# --- Pre-requisites ---
smart_install("tqdm", "tqdm")
smart_install("gdown", "gdown")

# --- Step 1: Repository ---
set_progress(5, "Step 1/5: Cloning repository...")
if not os.path.exists('/content/CryoAtom'):
    subprocess.run("git clone https://github.com/YangLab-SDU/CryoAtom.git -q", shell=True, capture_output=True)

# --- Step 2: Dependencies ---
deps = [
    ("Bio", "biopython==1.81"), ("mrcfile", "mrcfile==1.4.3"), ("einops", "einops==0.6.0"),
    ("esm", "fair-esm==2.0.0"), ("pyhmmer", "pyhmmer==0.7.1"), ("ptflops", "ptflops==0.6.6"),
    ("rna_fm", "rna-fm==0.2.2"), ("openpyxl", "openpyxl"), ("joblib", "joblib==1.0.1"),
    ("pandas", "pandas==1.3.5"), ("matplotlib", "matplotlib==3.5.3"), ("sklearn", "scikit-learn==0.24.0"),
    ("py3Dmol", "py3Dmol")
]
for i, (imp_name, pkg_name) in enumerate(deps):
    progress = 10 + int((i / len(deps)) * 40)
    set_progress(progress, f"Step 2/5: Installing {pkg_name}...")
    smart_install(imp_name, pkg_name)

# --- Step 3: Core ---
set_progress(55, "Step 3/5: Installing CryoAtom2 core...")
if importlib.util.find_spec("CryoAtom2") is None:
    %cd /content/CryoAtom
    subprocess.run("python setup.py install", shell=True, capture_output=True)
    %cd /content

# --- Step 4: Weights ---
set_progress(65, "Step 4/5: Handling Model Weights...")
target_a = "/content/CryoAtom/CryoAtom2/checkpoint"
target_b = "/root/.cache/torch/hub/checkpoints"
os.makedirs(target_a, exist_ok=True)
os.makedirs(target_b, exist_ok=True)

group_a = ["RUNet.pth", "CryoNet.pth", "CryoNet_no_seq.pth"]
group_b = ["esm2_t33_650M_UR50D-contact-regression.pt", "esm2_t33_650M_UR50D.pt", "RNA-FM_pretrained.pth"]

if Use_Google_Drive:
    set_progress(70, "Step 4/5: Mounting Google Drive...")
    try:
        drive.mount('/content/drive', force_remount=True)
        drive_path = "/content/drive/MyDrive/CryoAtom_Models_Pack"
        if os.path.exists(drive_path):
            for filename in group_a:
                src, dst = os.path.join(drive_path, filename), os.path.join(target_a, filename)
                if os.path.exists(src):
                    if os.path.exists(dst): os.remove(dst)
                    os.symlink(src, dst)
            for filename in group_b:
                src, dst = os.path.join(drive_path, filename), os.path.join(target_b, filename)
                if os.path.exists(src):
                    if os.path.exists(dst): os.remove(dst)
                    os.symlink(src, dst)
            set_progress(90, "Step 4/5: Weights linked from Drive.")
        else:
            set_progress(70, "Step 4/5: Drive pack missing. Downloading...", "#f39c12")
            import gdown
            gdown.download_folder(id="1BUWOsEXYyX85ClZvNjzeTCaTKQd9UVF9", output=target_a, quiet=True, remaining_ok=True)
    except:
        set_progress(70, "Step 4/5: Drive error. Downloading instead...", "#f39c12")
        import gdown
        gdown.download_folder(id="1BUWOsEXYyX85ClZvNjzeTCaTKQd9UVF9", output=target_a, quiet=True, remaining_ok=True)
else:
    set_progress(70, "Step 4/5: Downloading weights (~2GB)...")
    import gdown
    gdown.download_folder(id="1BUWOsEXYyX85ClZvNjzeTCaTKQd9UVF9", output=target_a, quiet=True, remaining_ok=True)

for root_dir, _, files in os.walk(target_a, topdown=False):
    for f in files:
        src_file = os.path.join(root_dir, f)
        if f in group_a and src_file != os.path.join(target_a, f): os.replace(src_file, os.path.join(target_a, f))
        elif f in group_b: os.replace(src_file, os.path.join(target_b, f))
    if root_dir != target_a and not os.listdir(root_dir): os.rmdir(root_dir)

# --- Step 5: Validation ---
set_progress(95, "Step 5/5: Final validation...")
%cd /content/CryoAtom
getp_path = "/content/CryoAtom/CryoAtom2/utils/getp"
if os.path.exists(getp_path): os.chmod(getp_path, 0o755)

final_check = ["Bio", "mrcfile", "pyhmmer", "esm", "CryoAtom2"]
missing = [m for m in final_check if importlib.util.find_spec(m) is None]

if not missing:
    set_progress(100, "Setup Completed Successfully!", "#f39c12")
else:
    set_progress(100, f"Finished with warnings. Missing: {missing}", "#e74c3c")

/content/CryoAtom
/content
/content/CryoAtom


In [ ]:
#@title # Step 3: Modeling { display-mode: "form" }
import os
import subprocess

def run_task():
    global final_cmd

    if not final_cmd:
        print("Error: No command found. Please finish Step 1 first!")
        return

    print(f"Execution Started...")
    print(f"CMD: {final_cmd}")
    print("-" * 60)
    %cd /content/CryoAtom
    process = subprocess.Popen(final_cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="")

    process.wait()

    if process.returncode == 0:
        print(f"\nTask finished successfully.")
    else:
        print(f"\nTask failed with return code {process.returncode}")

run_task()

In [16]:
#@title # Step 4-A: 3D Structure Preview { display-mode: "form" }
#@markdown 选择着色模式并运行单元格以显示结构。

Color_Mode = "Chain" #@param ["Confidence", "Residue", "Chain"]
import py3Dmol, os, re

# 1. 自动定位路径 (保持原有逻辑)
cmd = globals().get('final_cmd', "")
res_path = None
if cmd:
    try:
        j_dir = re.search(r'-o\s+([^\s\'"]+)', cmd).group(1).strip("'\"")
        b_name = j_dir.split('/')[-1]
        res_path = os.path.join(j_dir, f"{b_name}.cif")
    except: pass
if not res_path or not os.path.exists(res_path):
    cifs = sorted([f for f in os.listdir('.') if f.endswith('.cif')], key=os.path.getmtime)
    res_path = cifs[-1] if cifs else None

if not res_path:
    print("Error: No result file (.cif) found. Please run Step 3 first.")
else:
    # 2. 渲染逻辑
    with open(res_path, 'r') as f:
        data = f.read()

    view = py3Dmol.view(width=900, height=600)
    # 明确告知以 cif 格式加载
    view.addModel(data, 'cif')

    if Color_Mode == "Confidence":
        # 置信度：直接映射 B-factor (pLDDT)
        # 范围 50-90 是区分低置信度(红)到高置信度(蓝)的标准
        view.setStyle({'cartoon': {'colorscheme': {'prop': 'b', 'gradient': 'rdylbu', 'min': 50, 'max': 90}}})

    elif Color_Mode == "Residue":
        # 残基：使用 rainbow 渐变，确保从 N 端到 C 端呈现彩虹色
        view.setStyle({'cartoon': {'color': 'spectrum'}})

    elif Color_Mode == "Chain":
        # 链：解决灰色问题的关键。改用 'chain' 或手动指定 scheme
        # 对于 CIF，'chain' 方案比 'chainid' 更具兼容性
        view.setStyle({'cartoon': {'colorscheme': 'chain'}})
        # 补充：如果上面失败，尝试针对每个原子强制应用链颜色
        view.setStyle({'model': -1}, {'cartoon': {'colorscheme': 'chain'}})

    view.zoomTo()
    view.setBackgroundColor('#ffffff')

    # 打印状态
    print(f"File: {os.path.basename(res_path)}")
    print(f"Coloring Mode: {Color_Mode}")

    view.show()

File: Example_7xht.cif
Coloring Mode: Chain (B-factor Mapping)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.